In [2]:
import pandas as pd

%store -r full_pop_test
%store -r full_pop_df
%store -r claim_test

In [3]:
full_pop_test["pure_premium"] = full_pop_test["pred_freq"] * full_pop_test["pred_sev"]
full_pop_test["pure_premium"].describe()

count    135603.000000
mean         92.580589
std          75.962284
min           0.234504
25%          30.784506
50%          87.098425
75%         131.814916
max        2626.621673
Name: pure_premium, dtype: float64

In [4]:
full_pop_df["pure_premium"] = full_pop_df["pred_freq"] * full_pop_df["pred_sev"]
full_pop_df["pure_premium"].describe()

count    678013.000000
mean         92.150511
std          75.433775
min           0.185344
25%          30.414086
50%          86.790216
75%         131.629875
max        2756.627348
Name: pure_premium, dtype: float64

In [5]:
full_pop_df["age_band"] = pd.cut(
    full_pop_df["DrivAge"],
    bins=[18,25,35,45,55,65,100],
    labels=["18-25","26-35","36-45","46-55","56-65","66+"],
)

In [6]:
rating_cells = full_pop_df.groupby(["age_band", "Region"]).agg(
    avg_pure_premium=("pure_premium", "mean"),
    n_policies=("IDpol", "count"),
).reset_index()

In [7]:
base_class = rating_cells[(rating_cells["age_band"] == "46-55") & (rating_cells["Region"] == "R24")]
base_premium = base_class["avg_pure_premium"].values[0]

rating_cells["relativity"] = rating_cells["avg_pure_premium"] / base_premium
rating_cells["credibility_flag"] = rating_cells["n_policies"] < 250

In [8]:
age_relativity = rating_cells.groupby("age_band").apply(
    lambda g: pd.Series({
        "relativity": round((g["avg_pure_premium"] * g["n_policies"]).sum() / (g["n_policies"].sum() * base_premium),2),
        "n_policies": g["n_policies"].sum(),
        "credibility_flag": g["n_policies"].sum() < 2000,
    })
).reset_index()

print(age_relativity)

  age_band  relativity  n_policies  credibility_flag
0    18-25        1.29       38147             False
1    26-35        1.00      149183             False
2    36-45        0.93      170352             False
3    46-55        0.99      161899             False
4    56-65        1.10       90691             False
5      66+        1.29       66993             False


In [9]:
region_relativity = rating_cells.groupby("Region").apply(
    lambda g: pd.Series({
        "relativity": round((g["avg_pure_premium"] * g["n_policies"]).sum() / (g["n_policies"].sum() * base_premium),2),
        "n_policies": g["n_policies"].sum(),
        "credibility_flag": g["n_policies"].sum() < 2000,
    })
).reset_index()

print(region_relativity.sort_values("relativity", ascending=False))

   Region  relativity  n_policies  credibility_flag
21    R94        1.61        4511             False
12    R53        1.24       42087             False
20    R93        1.15       79246             False
17    R82        1.15       84652             False
0     R11        1.11       69762             False
5     R25        1.11       10874             False
6     R26        1.06       10482             False
4     R24        1.05      160322             False
2     R22        1.03        7989             False
9     R42        1.03        2199             False
16    R74        0.95        4565             False
11    R52        0.93       38717             False
13    R54        0.93       19010             False
19    R91        0.90       35763             False
8     R41        0.90       12983             False
7     R31        0.86       27260             False
14    R72        0.84       31298             False
1     R21        0.76        3026             False
10    R43   

In [10]:
def price_policy(driv_age, region, base_premium=base_premium,
                 age_relativity_table=age_relativity, region_relativity_table=region_relativity):
    if driv_age <= 25:
        age_band = "18-25"
    elif driv_age <= 35:
        age_band = "26-35"
    elif driv_age <= 45:
        age_band = "36-45"
    elif driv_age <= 55:
        age_band = "46-55"
    elif driv_age <= 65:
        age_band = "56-65"
    else:
        age_band = "66+"

    age_rel = age_relativity_table.loc[age_relativity_table["age_band"] == age_band, "relativity"].values[0]
    region_rel = region_relativity_table.loc[region_relativity_table["Region"] == region, "relativity"].values[0]

    premium = base_premium * age_rel * region_rel
    return round(premium, 2)

In [11]:
print(price_policy(driv_age=22, region="R94"))
print(price_policy(driv_age=50, region="R24"))
print(price_policy(driv_age=70, region="R83"))

184.41
92.3
64.14


In [13]:
customers = {
    "Age": [22, 50, 70],
    "Region": ["R94", "R24", "R83"],
    "Premium": [price_policy(driv_age=22, region="R94"), price_policy(driv_age=50, region="R24"), price_policy(driv_age=70, region="R83")],
}
print(pd.DataFrame(customers))

   Age Region  Premium
0   22    R94   184.41
1   50    R24    92.30
2   70    R83    64.14
